In [ ]:
# Figure 1 E, F, Figure 5 B, Supplemental Figure 5 E, F, G, H (beh rl modeling)

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.pyplot import cm

from paths import DATA_DIR, fig_dir


import statsmodels.api as sm
from statsmodels.stats.nonparametric import *
from scipy.stats import mannwhitneyu

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 

In [ ]:
# shared schema
from spyglass.common import *

#custom schema
from find_my_data import *
from plot_content import *
from plot_rlmodel import *
from fig_helpers import *
from alison_rlmodel import *

In [ ]:
set_figure_defaults()

fig_path = fig_dir('figs26')
if not os.path.exists(fig_path):
    os.makedirs(fig_path)

save_fig = False

subject_ids = ['j16', 'chimi', 'senor', 'wilbur', 'peanut']
subject_ids_and_all = ['j16', 'chimi', 'senor', 'wilbur', 'peanut','all_rats']

#### load data

In [ ]:
# find neural data files
stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')
    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]
print(stable_clusterless_nwbs)

In [ ]:
behavior_model_params_name = "beta_stable_withleaf"
# find model data with neural data
beta_results = {}
beta_results_stable = {}
# get stable only days
for subject_id in subject_ids:
    df = (BehaviorModelResults() & {'behavior_model_params_name': behavior_model_params_name, 'subject_id': subject_id}).fetch1_dataframe()
    beta_results[subject_id] = df
    stable_nwbs = stable_clusterless_nwbs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_nwbs)]
    beta_results_stable[subject_id] = df_stable[np.logical_and(df_stable['contingency']!=100100100100100100, df_stable['contingency']!=505050505050)]
beta_results_stable['all_rats'] = pd.concat([beta_results_stable[subject_id] for subject_id in subject_ids])

In [ ]:
n_total=0
for subject_id in subject_ids:
    print(f'{subject_id}, n= {len(beta_results_stable[subject_id])} trials')
    print(f'   SWITCH: {len(beta_results_stable[subject_id][beta_results_stable[subject_id].stemswitch==1])}')
    print(f'   Stay:   {len(beta_results_stable[subject_id][beta_results_stable[subject_id].stemswitch==0])}')
    n_total += len(beta_results_stable[subject_id])
print(n_total)

#### functions

In [ ]:
def add_ak_style_cols_to_RL_output(df, subject_id):
    # assuming:
    #   df has exactly the columns listed from Model A, including:
    #   'daynum', 'daysessionnum', 'trial_number_by_epoch',
    #   'stemchoice', 'leafchoice',
    #   betadist_α1..6, betadist_β1..6, betadist_var1..6,
    #   plus all the Q*, depletion*, *_variance columns, etc.
    
    # Sort in chronological order
    trial_order_cols = ["daynum", "daysessionnum", "trial_number_by_epoch"]
    df = df.sort_values(trial_order_cols).reset_index(drop=True)

    session_key = "daysessionnum"   # used for session-respecting variants
    
    # betadist_mu1..6 = alpha / (alpha + beta)  (Model B-only)
    for i in range(1, 7):
        a = df[f"betadist_α{i}"]
        b = df[f"betadist_β{i}"]
        df[f"betadist_mu{i}"] = a / (a + b)

    #GLOBAL (Model B-style) per-leaf deltas and ratios
    # match Julia Model B: deltas and ratios along the full sequence
    for i in range(1, 7):
        mu_col  = f"betadist_mu{i}"
        var_col = f"betadist_var{i}"

        # delta mu_t = mu_t − mu_{t−1}, row 0 = 0 (as in Model B)
        mu_delta = df[mu_col].diff()
        mu_delta.iloc[0] = 0.0
        df[f"betadist_mu{i}_delta"] = mu_delta

        # delta var_t = var_t − var_{t−1}, row 0 = 0
        var_delta = df[var_col].diff()
        var_delta.iloc[0] = 0.0
        df[f"betadist_var{i}_delta"] = var_delta

        # var_ratio_t = var_t / var_{t−1}, row 0 = 0
        prev_var = df[var_col].shift(1)
        var_ratio = pd.Series(0.0, index=df.index)
        var_ratio.iloc[1:] = df[var_col].iloc[1:].values / prev_var.iloc[1:].values
        df[f"betadist_var{i}_ratio"] = var_ratio

    #GLOBAL (Model B-style) global change metrics
    mu_delta_cols  = [f"betadist_mu{i}_delta"  for i in range(1, 7)]
    var_delta_cols = [f"betadist_var{i}_delta" for i in range(1, 7)]

    df["prev_global_mu_delta"] = df[mu_delta_cols].sum(axis=1)
    df["prev_global_mu_delta_abs"] = df[mu_delta_cols].abs().sum(axis=1)

    df["prev_global_var_delta"] = df[var_delta_cols].sum(axis=1)
    df["prev_global_var_delta_abs"] = df[var_delta_cols].abs().sum(axis=1)
    
    #GLOBAL leaf-aligned features (prev/next/upcoming) as in Model B
    n = len(df)
    # leaf index: 1..6
    leaf_index_global = (df["stemchoice"] - 1) * 2 + df["leafchoice"]

    # "upcoming" leaf index on the same stem, opposite leaf
    upcoming_index_global = (df["stemchoice"] - 1) * 2 + (3 - df["leafchoice"])

    #allocate Model B-only leaf-aligned columns
    for col in [
        "prev_leaf_mu_delta", "prev_leaf_var_delta", "prev_leaf_var_ratio",
        "next_choice_leaf_mu_delta", "next_choice_leaf_var_delta", "next_choice_leaf_var_ratio",
        "upcoming_leaf_mu_delta", "upcoming_leaf_var_delta", "upcoming_leaf_var_ratio",
    ]:
        df[col] = 0.0 #model B initializes all rows to 0.0

    for t in range(n):
        s = int(df.loc[t, "stemchoice"])
        l = int(df.loc[t, "leafchoice"])
        prevleaf = int((s - 1) * 2 + l)

        # previous-leaf deltas/ratio at trial t
        df.loc[t, "prev_leaf_mu_delta"]  = df.loc[t, f"betadist_mu{prevleaf}_delta"]
        df.loc[t, "prev_leaf_var_delta"] = df.loc[t, f"betadist_var{prevleaf}_delta"]
        df.loc[t, "prev_leaf_var_ratio"] = df.loc[t, f"betadist_var{prevleaf}_ratio"]

        # next-choice and upcoming only if there is a next row
        if t < n - 1:
            # next chosen leaf is leaf_index_global at t+1
            nextleaf = int(leaf_index_global.iloc[t+1])

            df.loc[t, "next_choice_leaf_mu_delta"]  = df.loc[t+1, f"betadist_mu{nextleaf}_delta"]
            df.loc[t, "next_choice_leaf_var_delta"] = df.loc[t+1, f"betadist_var{nextleaf}_delta"]
            df.loc[t, "next_choice_leaf_var_ratio"] = df.loc[t+1, f"betadist_var{nextleaf}_ratio"]

            # upcoming leaf on current stem at t, evaluated at t+1
            upcomingleaf = int(upcoming_index_global.iloc[t])
            df.loc[t, "upcoming_leaf_mu_delta"]  = df.loc[t+1, f"betadist_mu{upcomingleaf}_delta"]
            df.loc[t, "upcoming_leaf_var_delta"] = df.loc[t+1, f"betadist_var{upcomingleaf}_delta"]
            df.loc[t, "upcoming_leaf_var_ratio"] = df.loc[t+1, f"betadist_var{upcomingleaf}_ratio"]
    
    # SESSION-RESPECTING variants (suffix _sess)
    # These do NOT overwrite anything they're additional cols

    # Per-leaf deltas and ratios within-session
    for i in range(1, 7):
        mu_col  = f"betadist_mu{i}"
        var_col = f"betadist_var{i}"

        df[f"betadist_mu{i}_delta_sess"] = df.groupby(session_key)[mu_col].diff()
        df[f"betadist_var{i}_delta_sess"] = df.groupby(session_key)[var_col].diff()
        prev_var_sess = df.groupby(session_key)[var_col].shift(1)
        df[f"betadist_var{i}_ratio_sess"] = df[var_col] / prev_var_sess

    mu_delta_sess_cols  = [f"betadist_mu{i}_delta_sess"  for i in range(1, 7)]
    var_delta_sess_cols = [f"betadist_var{i}_delta_sess" for i in range(1, 7)]

    df["prev_global_mu_delta_sess"] = df[mu_delta_sess_cols].sum(axis=1, min_count=1)
    df["prev_global_mu_delta_abs_sess"] = df[mu_delta_sess_cols].abs().sum(axis=1, min_count=1)
    df["prev_global_var_delta_sess"] = df[var_delta_sess_cols].sum(axis=1, min_count=1)
    df["prev_global_var_delta_abs_sess"] = df[var_delta_sess_cols].abs().sum(axis=1, min_count=1)


    # SESSION-RESPECTING leaf-index neighbor relationships
    leaf_index_sess = (df["stemchoice"] - 1) * 2 + df["leafchoice"]
    leaf_index_next_sess = leaf_index_sess.groupby(df[session_key]).shift(-1)

    upcoming_index_sess = (df["stemchoice"] - 1) * 2 + (3 - df["leafchoice"])
    upcoming_index_next_sess = upcoming_index_sess.groupby(df[session_key]).shift(-1)

    for col in [
        "prev_leaf_mu_delta_sess", "prev_leaf_var_delta_sess", "prev_leaf_var_ratio_sess",
        "next_choice_leaf_mu_delta_sess", "next_choice_leaf_var_delta_sess",
        "next_choice_leaf_var_ratio_sess",
        "upcoming_leaf_mu_delta_sess", "upcoming_leaf_var_delta_sess",
        "upcoming_leaf_var_ratio_sess",
        "next_choice_variance_sess", "upcoming_leaf_variance_sess",
    ]:
        df[col] = np.nan

    for t in range(n):
        s = int(df.loc[t, "stemchoice"])
        l = int(df.loc[t, "leafchoice"])
        prevleaf = int((s - 1) * 2 + l)

        # previous leaf deltas/ratio within-session
        df.loc[t, "prev_leaf_mu_delta_sess"]  = df.loc[t, f"betadist_mu{prevleaf}_delta_sess"]
        df.loc[t, "prev_leaf_var_delta_sess"] = df.loc[t, f"betadist_var{prevleaf}_delta_sess"]
        df.loc[t, "prev_leaf_var_ratio_sess"] = df.loc[t, f"betadist_var{prevleaf}_ratio_sess"]

        # within-session next-choice
        leaf_next_s = leaf_index_next_sess.iloc[t]
        if not pd.isna(leaf_next_s):
            leaf_next_s = int(leaf_next_s)
            df.loc[t, "next_choice_variance_sess"] = df.loc[t+1, f"betadist_var{leaf_next_s}"]
            df.loc[t, "next_choice_leaf_mu_delta_sess"]  = df.loc[t+1, f"betadist_mu{leaf_next_s}_delta_sess"]
            df.loc[t, "next_choice_leaf_var_delta_sess"] = df.loc[t+1, f"betadist_var{leaf_next_s}_delta_sess"]
            df.loc[t, "next_choice_leaf_var_ratio_sess"] = df.loc[t+1, f"betadist_var{leaf_next_s}_ratio_sess"]

        # within-session upcoming leaf
        leaf_up_next_s = upcoming_index_next_sess.iloc[t]
        if not pd.isna(leaf_up_next_s):
            leaf_up_next_s = int(leaf_up_next_s)
            df.loc[t, "upcoming_leaf_variance_sess"]      = df.loc[t+1, f"betadist_var{leaf_up_next_s}"]
            df.loc[t, "upcoming_leaf_mu_delta_sess"]      = df.loc[t+1, f"betadist_mu{leaf_up_next_s}_delta_sess"]
            df.loc[t, "upcoming_leaf_var_delta_sess"]     = df.loc[t+1, f"betadist_var{leaf_up_next_s}_delta_sess"]
            df.loc[t, "upcoming_leaf_var_ratio_sess"]     = df.loc[t+1, f"betadist_var{leaf_up_next_s}_ratio_sess"]
    return df

def set_nan_at_max(group):
    max_test_series = group['trials_to_next_switch_groups'].max()
    if group['stem_switch'][-1:].values[0] == False: # if final trial of epoch isn't a switch, then last group cant be countign down to next switch, so then replace with nans
        group['trials_to_next_switch_with_nans'] = group.apply(lambda row: np.nan if row['trials_to_next_switch_groups'] == max_test_series else row['trials_to_next_switch'], axis=1)
    else:
        group['trials_to_next_switch_with_nans'] = group.apply(lambda row: row['trials_to_next_switch'] if row['trials_to_next_switch_groups'] == max_test_series else row['trials_to_next_switch'], axis=1)
#         print(f'max test group: {max_test_series} for nwb_file_name {group["nwb_file_name"][0].values[0]} and epoch {group["epoch"][0].values[0]}. Because final trial is a a stay trial, last group for trials to next switch can be nans.')
#     elif group['stem_switch'][-1:].values[0] == True:
#         print(group['stem_switch'][-1:])
#         print('Final trial is a switch, so trials to next switch does NOT need to be a nan')
    return group

# Add own stem switch based on groups
def add_periswitch_cols(beta_dfs, subject_id):
    beta_dfs[subject_id]['SstemOption'] = beta_dfs[subject_id].groupby(by = ['nwb_file_name', 'epoch',])['stem'].shift(1)
    beta_dfs[subject_id]['stem_switch'] = (beta_dfs[subject_id]['SstemOption']!=beta_dfs[subject_id]['stem']) # this is functionally within epoch

    # add the trial info cols
    beta_dfs[subject_id]['trials_from_prior_switch_groups'] = beta_dfs[subject_id].groupby(by = ['nwb_file_name','epoch'])['stem_switch'].cumsum()
    beta_dfs[subject_id]['trials_from_prior_switch'] = beta_dfs[subject_id].groupby(by = ['nwb_file_name','epoch','trials_from_prior_switch_groups']).cumcount()
    beta_dfs[subject_id]['trials_from_prior_switch'] = beta_dfs[subject_id]['trials_from_prior_switch'].where(beta_dfs[subject_id]['trials_from_prior_switch_groups'].gt(0), np.nan)


    beta_dfs[subject_id]['trials_to_next_switch_groups'] = beta_dfs[subject_id].groupby(by = ['nwb_file_name','epoch']).apply(lambda x_df: x_df['stem_switch'].shift(fill_value=False).cumsum()).reset_index(name='group').set_index('level_2')['group'] #set_index('id')['group']
    beta_dfs[subject_id]['trials_to_next_switch'] = beta_dfs[subject_id].groupby(by=['nwb_file_name','epoch','trials_to_next_switch_groups'])['trials_to_next_switch_groups'].cumcount(ascending=False)

    beta_dfs[subject_id] = beta_dfs[subject_id].groupby(by=['nwb_file_name','epoch']).apply(set_nan_at_max)
    beta_dfs[subject_id]['trials_from_next_switch'] = beta_dfs[subject_id]['trials_to_next_switch_with_nans']*-1

    beta_dfs[subject_id]['try_bout_idx'] = beta_dfs[subject_id].groupby(by = ['nwb_file_name','epoch'])['stem'].transform(lambda x_df: (x_df != x_df.shift(1)).cumsum())
    beta_dfs[subject_id]['bout_len'] = beta_dfs[subject_id].groupby(by = ['nwb_file_name','epoch', 'try_bout_idx'])['try_bout_idx'].transform(len)
    beta_dfs[subject_id]['bout_len_new'] = beta_dfs[subject_id].groupby(by = ['nwb_file_name','epoch', 'try_bout_idx'])['trials_from_prior_switch'].transform(lambda x: np.max(x))
    return beta_dfs[subject_id]

def add_dvs(df, session_key="daysessionnum"):
    """
    Timing convention:
      - Row t contains PRE-outcome belief state for trial t
      - Outcome on trial t is incorporated into betadist_* on row t+1

    Adding (all session-respecting):
      Trial t, chosen leaf:
        - chosen_leaf_var_preoutcome_sess
        - chosen_leaf_var_postoutcome_sess

      Trial t, starting-patch sibling leaf
        (sibling of leaf chosen on trial t-1):
        - startpatch_sibling_var_preoutcome_sess
        - startpatch_sibling_var_postoutcome_sess

      Trial t, global environment uncertainty (pre-outcome):
        - global_leaf_var_sum_preoutcome_sess

      Trial t, mu update magnitudes and signed updates due to trial t outcome
      (computed from mu deltas on row t+1 and aligned back onto row t):
        Absolute:
          - global_mu_update_abs_postoutcome_sess
          - chosen_leaf_mu_update_abs_postoutcome_sess
          - chosen_patch_mu_update_abs_postoutcome_sess
          - unchosen_patches_mu_update_abs_postoutcome_sess
        Signed:
          - global_mu_update_signed_postoutcome_sess
          - chosen_leaf_mu_update_signed_postoutcome_sess
          - chosen_patch_mu_update_signed_postoutcome_sess
          - unchosen_patches_mu_update_signed_postoutcome_sess

      Trial t, variance update magnitudes and signed updates due to trial t outcome
      (computed from var deltas on row t+1 and aligned back onto row t):
        Absolute:
          - global_var_update_abs_postoutcome_sess
          - chosen_leaf_var_update_abs_postoutcome_sess
          - chosen_patch_var_update_abs_postoutcome_sess
          - unchosen_patches_var_update_abs_postoutcome_sess
        Signed:
          - global_var_update_signed_postoutcome_sess
          - chosen_leaf_var_update_signed_postoutcome_sess
          - chosen_patch_var_update_signed_postoutcome_sess
          - unchosen_patches_var_update_signed_postoutcome_sess
    """

    n = len(df)

    chosen_leaf = (df["stemchoice"] - 1) * 2 + df["leafchoice"]
    start_leaf = chosen_leaf.groupby(df[session_key]).shift(1)

    startpatch_sibling = start_leaf.copy()
    m = ~startpatch_sibling.isna()
    startpatch_sibling.loc[m] = startpatch_sibling.loc[m].apply(
        lambda x: int(x) + 1 if int(x) % 2 == 1 else int(x) - 1
    )

    var_cols = [f"betadist_var{i}" for i in range(1, 7)]
    df["global_leaf_var_sum_preoutcome_sess"] = df[var_cols].sum(axis=1)

    mu_delta_cols = [f"betadist_mu{i}_delta_sess" for i in range(1, 7)]
    missing_mu_delta = [c for c in mu_delta_cols if c not in df.columns]
    if missing_mu_delta:
        raise KeyError(f"Missing required mu-delta columns: {missing_mu_delta}")

    var_delta_cols = [f"betadist_var{i}_delta_sess" for i in range(1, 7)]
    missing_var_delta = [c for c in var_delta_cols if c not in df.columns]
    if missing_var_delta:
        raise KeyError(f"Missing required var-delta columns: {missing_var_delta}")

    df["chosen_leaf_var_preoutcome_sess"] = np.nan
    df["chosen_leaf_var_postoutcome_sess"] = np.nan
    df["startpatch_sibling_var_preoutcome_sess"] = np.nan
    df["startpatch_sibling_var_postoutcome_sess"] = np.nan

    df["global_mu_update_abs_postoutcome_sess"] = np.nan
    df["chosen_leaf_mu_update_abs_postoutcome_sess"] = np.nan
    df["chosen_patch_mu_update_abs_postoutcome_sess"] = np.nan
    df["unchosen_patches_mu_update_abs_postoutcome_sess"] = np.nan
    df["global_mu_update_signed_postoutcome_sess"] = np.nan
    df["chosen_leaf_mu_update_signed_postoutcome_sess"] = np.nan
    df["chosen_patch_mu_update_signed_postoutcome_sess"] = np.nan
    df["unchosen_patches_mu_update_signed_postoutcome_sess"] = np.nan

    df["global_var_update_abs_postoutcome_sess"] = np.nan
    df["chosen_leaf_var_update_abs_postoutcome_sess"] = np.nan
    df["chosen_patch_var_update_abs_postoutcome_sess"] = np.nan
    df["unchosen_patches_var_update_abs_postoutcome_sess"] = np.nan
    df["global_var_update_signed_postoutcome_sess"] = np.nan
    df["chosen_leaf_var_update_signed_postoutcome_sess"] = np.nan
    df["chosen_patch_var_update_signed_postoutcome_sess"] = np.nan
    df["unchosen_patches_var_update_signed_postoutcome_sess"] = np.nan

    for t in range(n):
        cl = chosen_leaf.iloc[t]

        df.loc[t, "chosen_leaf_var_preoutcome_sess"] = df.loc[t, f"betadist_var{int(cl)}"]

        has_next = (t < n - 1) and (df.loc[t, session_key] == df.loc[t+1, session_key])
        if has_next:
            df.loc[t, "chosen_leaf_var_postoutcome_sess"] = df.loc[t+1, f"betadist_var{int(cl)}"]

        sl = startpatch_sibling.iloc[t]
        if not pd.isna(sl):
            df.loc[t, "startpatch_sibling_var_preoutcome_sess"] = df.loc[t, f"betadist_var{int(sl)}"]
            if has_next:
                df.loc[t, "startpatch_sibling_var_postoutcome_sess"] = df.loc[t+1, f"betadist_var{int(sl)}"]

        if has_next:
            dmu_next = df.loc[t+1, mu_delta_cols].astype(float).values
            dvar_next = df.loc[t+1, var_delta_cols].astype(float).values

            stem = int(df.loc[t, "stemchoice"])
            stem_idx = [2 * stem - 2, 2 * stem - 1]
            unchosen_idx = [i for i in range(6) if i not in stem_idx]
            chosen_leaf_idx = int(cl) - 1

            df.loc[t, "global_mu_update_abs_postoutcome_sess"] = np.abs(dmu_next).sum()
            df.loc[t, "global_mu_update_signed_postoutcome_sess"] = dmu_next.sum()

            df.loc[t, "chosen_leaf_mu_update_abs_postoutcome_sess"] = abs(dmu_next[chosen_leaf_idx])
            df.loc[t, "chosen_leaf_mu_update_signed_postoutcome_sess"] = dmu_next[chosen_leaf_idx]

            df.loc[t, "chosen_patch_mu_update_abs_postoutcome_sess"] = np.abs(dmu_next[stem_idx]).sum()
            df.loc[t, "chosen_patch_mu_update_signed_postoutcome_sess"] = dmu_next[stem_idx].sum()

            df.loc[t, "unchosen_patches_mu_update_abs_postoutcome_sess"] = np.abs(dmu_next[unchosen_idx]).sum()
            df.loc[t, "unchosen_patches_mu_update_signed_postoutcome_sess"] = dmu_next[unchosen_idx].sum()

            df.loc[t, "global_var_update_abs_postoutcome_sess"] = np.abs(dvar_next).sum()
            df.loc[t, "global_var_update_signed_postoutcome_sess"] = dvar_next.sum()

            df.loc[t, "chosen_leaf_var_update_abs_postoutcome_sess"] = abs(dvar_next[chosen_leaf_idx])
            df.loc[t, "chosen_leaf_var_update_signed_postoutcome_sess"] = dvar_next[chosen_leaf_idx]

            df.loc[t, "chosen_patch_var_update_abs_postoutcome_sess"] = np.abs(dvar_next[stem_idx]).sum()
            df.loc[t, "chosen_patch_var_update_signed_postoutcome_sess"] = dvar_next[stem_idx].sum()

            df.loc[t, "unchosen_patches_var_update_abs_postoutcome_sess"] = np.abs(dvar_next[unchosen_idx]).sum()
            df.loc[t, "unchosen_patches_var_update_signed_postoutcome_sess"] = dvar_next[unchosen_idx].sum()

    return df

def add_switch_advantage(df, session_key="daysessionnum"):
    """
    compute trial-t decision variables evaluated PRE-outcome (row t):

    stay_value(t):
      - mu of the sibling leaf of the START leaf
      - start leaf = leaf chosen on trial t-1 (within session)

    best_other_patch_value(t):
      - maximum over the two OTHER stems of
        (mean mu of the two leaves in that stem)

    switch_advantage(t):
      - best_other_patch_value - stay_value

    Adds:
      - stay_sibling_mu_preoutcome_sess
      - best_other_patch_mean_mu_preoutcome_sess
      - switch_advantage_mu_preoutcome_sess
    """

    n = len(df)

    mu_cols = [f"betadist_mu{i}" for i in range(1, 7)]
    missing = [c for c in mu_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required mu columns: {missing}")

    # chosen leaf on trial t
    chosen_leaf = (df["stemchoice"] - 1) * 2 + df["leafchoice"]

    # start leaf and start stem for trial t = values from t-1 within session
    start_leaf_index = chosen_leaf.groupby(df[session_key]).shift(1)
    start_stem_index = df["stemchoice"].groupby(df[session_key]).shift(1)

    # sibling of start leaf (1↔2, 3↔4, 5↔6)
    startpatch_sibling_leaf_index = start_leaf_index.copy()
    valid_mask = ~startpatch_sibling_leaf_index.isna()
    startpatch_sibling_leaf_index.loc[valid_mask] = startpatch_sibling_leaf_index.loc[
        valid_mask
    ].apply(lambda x: int(x) + 1 if int(x) % 2 == 1 else int(x) - 1)

    df["stay_sibling_mu_preoutcome_sess"] = np.nan
    df["best_other_patch_mean_mu_preoutcome_sess"] = np.nan
    df["switch_advantage_mu_preoutcome_sess"] = np.nan

    for t in range(n):
        start_stem_for_trial = start_stem_index.iloc[t]
        sibling_leaf_of_start_patch = startpatch_sibling_leaf_index.iloc[t]

        if pd.isna(start_stem_for_trial) or pd.isna(sibling_leaf_of_start_patch):
            continue

        start_stem_for_trial = int(start_stem_for_trial)
        sibling_leaf_of_start_patch = int(sibling_leaf_of_start_patch)

        # value of staying (other leaf in starting patch)
        stay_mu = df.loc[t, f"betadist_mu{sibling_leaf_of_start_patch}"]

        # values of the two alternative patches
        other_stems = [1, 2, 3]
        other_stems.remove(start_stem_for_trial)

        def stem_mean_mu(stem_id):
            leaf_a = 2 * stem_id - 1
            leaf_b = 2 * stem_id
            return 0.5 * (
                df.loc[t, f"betadist_mu{leaf_a}"] +
                df.loc[t, f"betadist_mu{leaf_b}"]
            )

        other_patch_mean_mus = [stem_mean_mu(stem_id) for stem_id in other_stems]
        best_other_patch_mean_mu = max(other_patch_mean_mus)

        df.loc[t, "stay_sibling_mu_preoutcome_sess"] = stay_mu
        df.loc[t, "best_other_patch_mean_mu_preoutcome_sess"] = best_other_patch_mean_mu
        df.loc[t, "switch_advantage_mu_preoutcome_sess"] = best_other_patch_mean_mu - stay_mu

    return df

def add_stem_q_dvs(df, session_key="daysessionnum"):
    """
    Timing convention:
      - Row t contains PRE-outcome decision variables for trial t
      - Outcome on trial t is incorporated into Q variables on row t+1
      - Therefore, "update due to trial t outcome" is measured as a delta on row t+1
        and aligned back onto row t (within-session).

    Switch advantage variables (PRE-outcome, row t):
      These are based on the initiating (start) stem for trial t, which is stemchoice[t-1] within-session.
      For each stem-value family, we compare:
        - stay_value = value of the start stem on row t
        - best_other_value = max value among the two non-start stems on row t
        - switch_advantage = best_other_value - stay_value

      Families:
        - Qstem*: stem choice decision logits used for stem softmax; includes β scalings, γ2-weighting for stay,
                 depletion, and explicit bias additions (stay_bias, turn_bias, spatial_bias; delay_turn_bias handling).
        - Qstem*_nobias: same construction but without explicit bias additions; still includes β scalings, γ2, depletion.
        - Qstem*_pre_update_post_bias: bias-accounting variant constructed alongside choice computation; still pre-outcome
                 on row t; includes β scalings, γ2, depletion, and the choice-relevant bias structure.

    Update-magnitude variables (POST-outcome effect of trial t, aligned onto row t):
      These are based on the destination/choice on trial t (stemchoice[t]).
      For each family, computing within-session deltas across stems, take row t+1 deltas (effect of trial t outcome),
      and aggregate:
        - global update: sum over stems
        - chosen-stem update: stemchoice[t]
        - unchosen-stems update: the other two stems

      Families used for update metrics:
        - Qstem_nobias
        - Qstem_pre_update_post_bias

      Both absolute and signed versions are included
    """

    required_cols = (
        [f"Qstem{i}" for i in (1, 2, 3)] +
        [f"Qstem{i}_nobias" for i in (1, 2, 3)] +
        [f"Qstem{i}_pre_update_post_bias" for i in (1, 2, 3)] +
        ["stemchoice", session_key]
    )
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    n = len(df)

    # initiating (start) stem for trial t is stemchoice[t-1] within session
    start_stem_index = df["stemchoice"].groupby(df[session_key]).shift(1)

    # within-session deltas for stem-value families used for update metrics
    for stem_id in (1, 2, 3):
        df[f"Qstem{stem_id}_nobias_delta_sess"] = (
            df.groupby(session_key)[f"Qstem{stem_id}_nobias"].diff()
        )
        df[f"Qstem{stem_id}_pre_update_post_bias_delta_sess"] = (
            df.groupby(session_key)[f"Qstem{stem_id}_pre_update_post_bias"].diff()
        )

    # switch advantage columns for three families
    for family in ("Qstem", "Qstem_nobias", "Qstem_pre_update_post_bias"):
        df[f"start_stem_value_preoutcome_sess_{family}"] = np.nan
        df[f"best_other_stem_value_preoutcome_sess_{family}"] = np.nan
        df[f"switch_advantage_preoutcome_sess_{family}"] = np.nan

    # update-magnitude columns for two families
    for family_tag in ("Qstem_nobias", "Qstem_pre_update_post_bias"):
        df[f"global_{family_tag}_update_abs_postoutcome_sess"] = np.nan
        df[f"chosen_stem_{family_tag}_update_abs_postoutcome_sess"] = np.nan
        df[f"unchosen_stems_{family_tag}_update_abs_postoutcome_sess"] = np.nan

        df[f"global_{family_tag}_update_signed_postoutcome_sess"] = np.nan
        df[f"chosen_stem_{family_tag}_update_signed_postoutcome_sess"] = np.nan
        df[f"unchosen_stems_{family_tag}_update_signed_postoutcome_sess"] = np.nan

    for t in range(n):
        has_next = (t < n - 1) and (df.loc[t, session_key] == df.loc[t+1, session_key])

        # switch advantage uses initiating (start) stem
        start_stem_for_trial = start_stem_index.iloc[t]
        if not pd.isna(start_stem_for_trial):
            start_stem_for_trial = int(start_stem_for_trial)
            non_start_stems = [1, 2, 3]
            non_start_stems.remove(start_stem_for_trial)

            for family in ("Qstem", "Qstem_nobias", "Qstem_pre_update_post_bias"):
                if family == "Qstem":
                    stay_value = df.loc[t, f"Qstem{start_stem_for_trial}"]
                    best_other_value = max(df.loc[t, f"Qstem{non_start_stems[0]}"],
                                           df.loc[t, f"Qstem{non_start_stems[1]}"])
                elif family == "Qstem_nobias":
                    stay_value = df.loc[t, f"Qstem{start_stem_for_trial}_nobias"]
                    best_other_value = max(df.loc[t, f"Qstem{non_start_stems[0]}_nobias"],
                                           df.loc[t, f"Qstem{non_start_stems[1]}_nobias"])
                else:  # Qstem_pre_update_post_bias
                    stay_value = df.loc[t, f"Qstem{start_stem_for_trial}_pre_update_post_bias"]
                    best_other_value = max(df.loc[t, f"Qstem{non_start_stems[0]}_pre_update_post_bias"],
                                           df.loc[t, f"Qstem{non_start_stems[1]}_pre_update_post_bias"])

                df.loc[t, f"start_stem_value_preoutcome_sess_{family}"] = stay_value
                df.loc[t, f"best_other_stem_value_preoutcome_sess_{family}"] = best_other_value
                df.loc[t, f"switch_advantage_preoutcome_sess_{family}"] = best_other_value - stay_value

        # update metrics use destination/choice stem for trial t, via deltas on row t+1
        if has_next:
            chosen_stem_for_trial = int(df.loc[t, "stemchoice"])
            unchosen_stems_for_trial = [1, 2, 3]
            unchosen_stems_for_trial.remove(chosen_stem_for_trial)

            d_next_nobias = np.array([
                float(df.loc[t+1, "Qstem1_nobias_delta_sess"]),
                float(df.loc[t+1, "Qstem2_nobias_delta_sess"]),
                float(df.loc[t+1, "Qstem3_nobias_delta_sess"]),
            ])

            d_next_preupd = np.array([
                float(df.loc[t+1, "Qstem1_pre_update_post_bias_delta_sess"]),
                float(df.loc[t+1, "Qstem2_pre_update_post_bias_delta_sess"]),
                float(df.loc[t+1, "Qstem3_pre_update_post_bias_delta_sess"]),
            ])

            for family_tag, d_next in (
                ("Qstem_nobias", d_next_nobias),
                ("Qstem_pre_update_post_bias", d_next_preupd),
            ):
                chosen_idx = chosen_stem_for_trial - 1
                unchosen_idx = [s - 1 for s in unchosen_stems_for_trial]

                df.loc[t, f"global_{family_tag}_update_abs_postoutcome_sess"] = np.abs(d_next).sum()
                df.loc[t, f"global_{family_tag}_update_signed_postoutcome_sess"] = d_next.sum()

                df.loc[t, f"chosen_stem_{family_tag}_update_abs_postoutcome_sess"] = abs(d_next[chosen_idx])
                df.loc[t, f"chosen_stem_{family_tag}_update_signed_postoutcome_sess"] = d_next[chosen_idx]

                df.loc[t, f"unchosen_stems_{family_tag}_update_abs_postoutcome_sess"] = np.abs(d_next[unchosen_idx]).sum()
                df.loc[t, f"unchosen_stems_{family_tag}_update_signed_postoutcome_sess"] = d_next[unchosen_idx].sum()

    return df

def get_glm_stay_switch(trialData, subject_id, biased, x_var1, x_var2, y_var, x_min=None, x_max=None):
    trialData_valid = trialData[np.logical_and(trialData[x_var1].notna(), trialData[y_var].notna()) & trialData[x_var2].notna()]

    # Assuming 'x' is independent variable and 'y' is binary dependent variable
    X = sm.add_constant(trialData_valid[[x_var1,x_var2]]) # Adding a constant for the intercept
    y = trialData_valid[y_var]

#     model = sm.Logit(y.astype(float), X.astype(float))
#     result = model.fit(disp=0)
        
    family = sm.families.Binomial()
    model = sm.GLM(y,
                    X,
                   family=family)
    result=model.fit()
    
    # Extract Coefficients, Pseudo R-squared, and p-values
    beta_0 = result.params['const']
    beta_1 = result.params[x_var1]
    beta_2 = result.params[x_var2]
    pseudo_r_squared = result.pseudo_rsquared(kind='cs')
    p_value = [result.pvalues[x_var1],result.pvalues[x_var2], result.pvalues['const']]
    aic = result.aic
    llf = result.llf
    coeffs = [beta_0, beta_1, beta_2, pseudo_r_squared, p_value, aic, llf]

    # Predictions for the Logistic Curve
    if (x_min is not None) & (x_max is not None):
        x_values1 = np.linspace(x_min, x_max, 300)
        x_values2 = np.linspace(x_min, x_max, 300)
        x_values = pd.DataFrame({x_var1:x_values1, x_var2:x_values2})
    else:
        x_values1 = np.linspace(trialData_valid[x_var1].min(), trialData_valid[x_var1].max(), 300)
        x_values2 = np.linspace(trialData_valid[x_var2].min(), trialData_valid[x_var2].max(), 300)
        x_values = pd.DataFrame({x_var1:x_values1, x_var2:x_values2})

    y_values = result.predict(sm.add_constant(x_values))
    
    # Equation and Statistics String
    equation_text = f'equation = f"log(p / (1 - p)) = {np.round(beta_0,2)} + {np.round(beta_1,2)}*X1 + {np.round(beta_2,2)}*X2\nPseudo R-squared: {pseudo_r_squared:.3f}\np-values: {p_value[0]:.3e},{p_value[1]:.3e}'
    
    return trialData_valid[[x_var1,x_var2]], trialData_valid[y_var], x_values, y_values, equation_text, coeffs

def find_other_leaf(row):
    if isinstance(row['Sleaves'], list):
        if  str(int(row['prior_leaf'])) == row['Sleaves'][0]:
            other_leaf = row['Sleaves'][1]
        elif  str(int(row['prior_leaf'])) == row['Sleaves'][1]:
            other_leaf = row['Sleaves'][0]
        else:
            raise Exception("Warning: looks like prior_leaf isnt in sleaves 0 or 1?")
    else:
        other_leaf = np.nan
    return other_leaf

def compute_stay_switch_values(row):
    leaves = range(1, 7)
    if row['SstemOption'] == 'A':
        Sleaves = ['1','2']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        GstemOption = ['B','C']
        SstemOptionBiased = '1'
        GstemOptionBiased = ['2','3']
    elif row['SstemOption'] == 'B':
        GstemOption = ['A','C']
        Sleaves = ['3','4']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        SstemOptionBiased = '2'
        GstemOptionBiased = ['1','3']
    elif row['SstemOption'] == 'C':
        GstemOption = ['A','B']
        Sleaves = ['5','6']
        Gleaves = [str(i) for i in leaves if i not in Sleaves]
        SstemOptionBiased = '3'
        GstemOptionBiased = ['1','2']
    else:
        # Default values if 'SstemOption' is not A, B, or C
        Sleaves, Gleaves, GstemOption, SstemOptionBiased, GstemOptionBiased = np.nan, np.nan, np.nan, np.nan, np.nan
    return Sleaves, Gleaves, GstemOption, SstemOptionBiased, GstemOptionBiased


def add_priorpatch_mu_updates(df, session_key="daysessionnum"):
    """
    where does the asymmetry in mu at unchosen patch come from?
    timing convention:
      - Row t contains PRE-outcome belief state for trial t
      - Outcome on trial t is incorporated into betadist_* on row t+1

    adding session-respecting vars, aligned to trial t using mu deltas on row t+1:
      Absolute mu update mags:
        - priorpatch_mu_update_abs_postoutcome_sess
        - other_unchosen_patch_mu_update_abs_postoutcome_sess

    Prior patch definition (within session):
      - On switch trials (stem_switch == True), prior patch is stemchoice on trial t-1
      - On stay trials, prior patch is carried forward from the most recent switch trial
      - First trial (and any trials before the first switch) remain NaN
    """

    n = len(df)

    mu_delta_cols = [f"betadist_mu{i}_delta_sess" for i in range(1, 7)]
    missing_mu_delta = [c for c in mu_delta_cols if c not in df.columns]
    if missing_mu_delta:
        raise KeyError(f"Missing required mu-delta columns: {missing_mu_delta}")

    if "stem_switch" not in df.columns:
        raise KeyError("Missing required column: stem_switch")

    df["priorpatch_mu_update_abs_postoutcome_sess"] = np.nan
    df["other_unchosen_patch_mu_update_abs_postoutcome_sess"] = np.nan

    prior_patch = pd.Series(np.nan, index=df.index, dtype=float)
    last_prior = np.nan

    for t in range(n):
        if t == 0 or df.loc[t, session_key] != df.loc[t - 1, session_key]:
            last_prior = np.nan
        else:
            if bool(df.loc[t, "stem_switch"]):
                last_prior = float(df.loc[t - 1, "stemchoice"])
        prior_patch.iloc[t] = last_prior

        has_next = (t < n - 1) and (df.loc[t, session_key] == df.loc[t + 1, session_key])
        if has_next and not pd.isna(prior_patch.iloc[t]):
            dmu_next = df.loc[t + 1, mu_delta_cols].astype(float).values

            stem = int(df.loc[t, "stemchoice"])
            pp = int(prior_patch.iloc[t])

            other_stem = list({1, 2, 3} - {stem, pp})
            if len(other_stem) == 1:
                pp_idx = [2 * pp - 2, 2 * pp - 1]
                other_idx = [2 * other_stem[0] - 2, 2 * other_stem[0] - 1]

                df.loc[t, "priorpatch_mu_update_abs_postoutcome_sess"] = np.abs(dmu_next[pp_idx]).sum()
                df.loc[t, "other_unchosen_patch_mu_update_abs_postoutcome_sess"] = np.abs(dmu_next[other_idx]).sum()

    return df

#### Build up dataframes with range of DVs

In [ ]:
beta_dfs = {}
beta_dfs_plus = {}
beta_dfs_plus_plus = {}
beta_dfs_plus_plus_plus = {}
beta_dfs_plus4 = {}

for subject_id in subject_ids:
    beta_dfs[subject_id] = add_ak_style_cols_to_RL_output(beta_results_stable[subject_id], subject_id)
    beta_dfs_plus[subject_id] = add_periswitch_cols(beta_dfs, subject_id)
    beta_dfs_plus_plus[subject_id] = add_dvs(beta_dfs_plus[subject_id])
    beta_dfs_plus_plus_plus[subject_id] = add_switch_advantage(beta_dfs_plus_plus[subject_id])
    beta_dfs_plus4[subject_id] = add_stem_q_dvs(beta_dfs_plus_plus_plus[subject_id])

beta_dfs_plus['all_rats'] = pd.concat([beta_dfs_plus[subject_id] for subject_id in subject_ids], ignore_index=True)
beta_dfs_plus_plus['all_rats'] = pd.concat([beta_dfs_plus_plus[subject_id] for subject_id in subject_ids], ignore_index=True)
beta_dfs_plus_plus_plus['all_rats'] = pd.concat([beta_dfs_plus_plus_plus[subject_id] for subject_id in subject_ids], ignore_index=True)
beta_dfs_plus4['all_rats'] = pd.concat([beta_dfs_plus4[subject_id] for subject_id in subject_ids], ignore_index=True)


#### F1 E

In [ ]:
df_all_dvs = beta_dfs_plus4.copy()

x_var1 = "stay_sibling_mu_preoutcome_sess"
x_var2 = "best_other_patch_mean_mu_preoutcome_sess"
y_var = "stem_switch"

In [ ]:
# try again to prep params and configure 

biased = False
choice = "switch_stay"

x_var1 = "stay_sibling_mu_preoutcome_sess"
x_var2 = "best_other_patch_mean_mu_preoutcome_sess"
y_var  = "stem_switch"  # change to "stemswitch" if that's your actual df column
subject_ids = ['all_rats', 'j16',  'chimi','senor', 'wilbur','peanut']
rats_in_order = ["j16", "chimi", "senor", "wilbur", "peanut"]
include_all_rats = True

colors = list(cm.tab20b([0, .8, .85, .1, .05]))

custom_ylim = True
show_all_rat_scatter = False

# -prep info to fit GLMs and collate coefficients 

if "all_rats" not in df_all_dvs:
    raise KeyError("df_all_dvs must contain key 'all_rats' for the concatenated dataframe.")

df_all = df_all_dvs["all_rats"].copy()

required_cols = [x_var1, x_var2, y_var, "daysessionnum"]
missing_cols = [c for c in required_cols if c not in df_all.columns]
if missing_cols:
    raise KeyError(f"df_all_dvs['all_rats'] missing required columns: {missing_cols}")

In [ ]:
# glm fit and track coefs
x_min = df_all[x_var2].min()
x_max = df_all[x_var2].max()

coeffs_collated = {}

subjects_to_fit = (["all_rats"] if include_all_rats else []) + rats_in_order

for subject_id in subjects_to_fit:
    if subject_id not in df_all_dvs:
        raise KeyError(f"df_all_dvs missing key '{subject_id}'.")

    trialData = df_all_dvs[subject_id].copy()
    trialData = trialData.dropna(subset=[x_var1, x_var2, y_var])

    x_trialData_valid, y_trialData_valid, x_values, y_values, equation_text, coeffs = get_glm_stay_switch(
        trialData,
        subject_id,
        biased,
        x_var1,
        x_var2,
        y_var,
        x_min=x_min,
        x_max=x_max,
    )

    key = (biased, choice, subject_id)
    coeffs_collated[key] = coeffs

In [ ]:
# get per-rat coeffs + pvals into plot_* dicts 
# coeffs = [beta_0, beta_1, beta_2, pseudo_r_squared, p_value, aic, llf]

plot_coeffs1 = {}
coeff_idx = 1
for (biased_k, choice_k, subject_id), coeffs in coeffs_collated.items():
    key = (biased_k, choice_k)
    plot_coeffs1.setdefault(key, [])
    if subject_id != "all_rats":
        plot_coeffs1[key].append(coeffs[coeff_idx])

plot_coeffs2 = {}
coeff_idx = 2
for (biased_k, choice_k, subject_id), coeffs in coeffs_collated.items():
    key = (biased_k, choice_k)
    plot_coeffs2.setdefault(key, [])
    if subject_id != "all_rats":
        plot_coeffs2[key].append(coeffs[coeff_idx])

plot_pvals1 = {}
coeff_idx = 4
for (biased_k, choice_k, subject_id), coeffs in coeffs_collated.items():
    key = (biased_k, choice_k)
    plot_pvals1.setdefault(key, [])
    if subject_id != "all_rats":
        plot_pvals1[key].append(coeffs[coeff_idx][0])

plot_pvals2 = {}
coeff_idx = 4
for (biased_k, choice_k, subject_id), coeffs in coeffs_collated.items():
    key = (biased_k, choice_k)
    plot_pvals2.setdefault(key, [])
    if subject_id != "all_rats":
        plot_pvals2[key].append(coeffs[coeff_idx][1])

# add intercept
plot_coeffs0 = {}
coeff_idx = 0
for (biased_k, choice_k, subject_id), coeffs in coeffs_collated.items():
    key = (biased_k, choice_k)
    plot_coeffs0.setdefault(key, [])
    if subject_id != "all_rats":
        plot_coeffs0[key].append(coeffs[coeff_idx])

plot_pvals0 = {}
coeff_idx = 4
for (biased_k, choice_k, subject_id), coeffs in coeffs_collated.items():
    key = (biased_k, choice_k)
    plot_pvals0.setdefault(key, [])
    if subject_id != "all_rats":
        plot_pvals0[key].append(coeffs[coeff_idx][2])

In [ ]:
# summary plot again
fig, axes = plt.subplots(ncols=1, nrows=1, figsize=(TWO_COLUMN/2,(TWO_COLUMN/4)*(1/GOLDEN_RATIO)))
np.random.seed(41)

# Stay-value beta (x position i)
for i, ((biased_k, choice_k), coeffs_values) in enumerate(plot_coeffs1.items()):
    mean_coeff = np.mean(coeffs_values)
    pvals = plot_pvals1[(biased_k, choice_k)]
    axes.bar(i, mean_coeff, alpha=.5, color="grey", width=.3)

    for rat_idx, (coeff, pval) in enumerate(zip(coeffs_values, pvals)):
        x_val = i + np.random.normal(0, .04)
        color_rat = colors[rat_idx]
        face_color = "none" if pval > .05 else color_rat
        axes.scatter(x_val, coeff, color=color_rat, marker="o", facecolors=face_color, s=40)

# Switch-value beta (x position i+0.5)
for i, ((biased_k, choice_k), coeffs_values) in enumerate(plot_coeffs2.items()):
    mean_coeff = np.mean(coeffs_values)
    pvals = plot_pvals2[(biased_k, choice_k)]
    axes.bar(i + .5, mean_coeff, alpha=.5, color="grey", width=.3)

    for rat_idx, (coeff, pval) in enumerate(zip(coeffs_values, pvals)):
        x_val = i + .5 + np.random.normal(0, .04)
        color_rat = colors[rat_idx]
        face_color = "none" if pval > .05 else color_rat

        if "subject_ids" in globals():
            label_text = f"Rat {subject_ids[rat_idx + 1][0].upper()}"
        else:
            label_text = f"Rat {rats_in_order[rat_idx].upper()}"
        axes.scatter(x_val, coeff, color=color_rat, marker="o", facecolors=face_color, s=30, label=label_text)

# Intercept (x position i-0.5)
for i, ((biased_k, choice_k), coeffs_values) in enumerate(plot_coeffs0.items()):
    mean_coeff = np.mean(coeffs_values)
    pvals = plot_pvals0[(biased_k, choice_k)]
    axes.bar(i - .5, mean_coeff, alpha=.5, color="grey", width=.3)

    axes.set_xticks([i - .5, i, i + .5])
    axes.set_xticklabels(["Constant", "Stay\nValue", "Switch\nValue"])
    axes.set_ylabel("Switch Choice Beta")

    for rat_idx, (coeff, pval) in enumerate(zip(coeffs_values, pvals)):
        x_val = i - .5 + np.random.normal(0, .04)
        color_rat = colors[rat_idx]
        face_color = color_rat #"none" if pval > .05 else color_rat
        axes.scatter(x_val, coeff, color=color_rat, marker="o", facecolors=face_color, s=30)

plt.axhline(0, color="black", lw=1)
plt.title(f"biases: {biased}", fontsize=8)
plt.legend(bbox_to_anchor=(1.1, .8), loc="upper left", frameon=False)
plt.tight_layout()
sns.despine(offset=5)

# Optional save
if save_fig:
    fig_name = f"allfilled_betabernoulli_glm_allrats_switchchoice_betas_stayswitch_intercept_x1_{x_var1}_x2_{x_var2}"
    plt.savefig(f"{fig_path}{fig_name}.pdf", format="pdf", bbox_inches="tight", pad_inches=.5, dpi=300)

fig.show()

In [ ]:
# Decide which subjects to print (matches prior logic)
subjects_to_print = (["all_rats"] if include_all_rats else []) + rats_in_order

rows = []
missing = []

for subject_id in subjects_to_print:
    key = (biased, choice, subject_id)
    if key not in coeffs_collated:
        missing.append(subject_id)
        continue

    # coeffs = [beta_0, beta_1, beta_2, pseudo_r_squared, p_values, aic, llf]
    coeffs = coeffs_collated[key]

    beta_intercept = coeffs[0]
    beta_stay = coeffs[1]
    beta_switch = coeffs[2]

    # p-values assumed to be ordered [stay, switch, intercept]
    pvals = coeffs[4]
    p_stay = pvals[0]
    p_switch = pvals[1]
    p_intercept = pvals[2]
    print(f"{subject_id} p stay, sw, intercept: {pvals}")

    rows.append({
        "subject_id": subject_id,
        "beta_intercept": beta_intercept,
        "p_intercept": p_intercept,
        "beta_stay_value": beta_stay,
        "p_stay_value": p_stay,
        "beta_switch_value": beta_switch,
        "p_switch_value": p_switch,
    })

df_betas = pd.DataFrame(rows)

# Add significance stars
def sigstars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

df_betas["sig_intercept"] = df_betas["p_intercept"].apply(sigstars)
df_betas["sig_stay_value"] = df_betas["p_stay_value"].apply(sigstars)
df_betas["sig_switch_value"] = df_betas["p_switch_value"].apply(sigstars)

# Round numeric columns for readability
num_cols = [
    "beta_intercept", "p_intercept",
    "beta_stay_value", "p_stay_value",
    "beta_switch_value", "p_switch_value",
]
df_betas[num_cols] = df_betas[num_cols].astype(float).round(4)

# Order columns nicely
df_betas = df_betas[
    [
        "subject_id",
        "beta_intercept", "p_intercept", "sig_intercept",
        "beta_stay_value", "p_stay_value", "sig_stay_value",
        "beta_switch_value", "p_switch_value", "sig_switch_value",
    ]
]

display(df_betas)

if missing:
    print("Missing subjects:", missing)


#### F1 F, SF5 E

In [ ]:
# F1 stats for relative switch value

dvs = ['switch_advantage_mu_preoutcome_sess',]
dv_descriptions = {'switch value - stay value',}   
   
                   
assert len(dvs) == len(dv_descriptions)
# dvs = ['delta_Qleaf_t_abs_1_ago','delta_Qpatch_t_abs_1_ago','delta_Qoutofpatch_t_abs_1_ago','delta_Qglobal_t_abs_1_ago'] # add 'delta_Qleaf_t_abs' 1 AGO!

trials_post_switch_max = 20 # positive int, inclusive
trials_pre_switch_min = -20 #n egative int, inclusive
exclude_switches = False

trial_subsets = ['trials_from_next_switch','trials_from_prior_switch']

markersize=3
err_style = 'bars'
ci=95

# withlegend = False

for dv in dvs:
    fig,ax = plt.subplots(ncols=6,nrows=1,figsize=(10*TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4), sharex=True, sharey=True)
    colors = list(cm.tab20b([0,.8, .85, .1, .05]))
#     colors += 'black'
    for i,subject_id in enumerate(beta_dfs_plus_plus_plus.keys()): #enumerate(subject_ids):
#         fig,ax = plt.subplots(ncols=1,nrows=1,figsize=(TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4))
#         print(subject_id, i)
#         print(i, subject_id)
        data_df = beta_dfs_plus_plus_plus[subject_id].copy()

        if np.logical_and(trials_post_switch_max is not None, trials_pre_switch_min is not None):
            data_df = data_df[np.logical_or(data_df['trials_from_prior_switch']<= trials_post_switch_max,
                                             data_df['trials_from_next_switch']>= trials_pre_switch_min)]
        elif trials_post_switch_max is not None:
            data_df = data_df[data_df['trials_from_prior_switch']<= trials_post_switch_max]
        elif trials_pre_switch_min is not None:
            data_df = data_df[data_df['trials_from_next_switch']>= trials_pre_switch_min]

        if exclude_switches:
            data_df = data_df[data_df['stem_switch']==False]

        for t,trial_subset in enumerate(trial_subsets):
            # important that the "and" step happens only on one trial subset at a time
            data_df_xlim = data_df[np.logical_and(data_df[trial_subset]>=trials_pre_switch_min,
                                                 data_df[trial_subset]<=trials_post_switch_max)]
            if subject_id != 'all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize, marker=None, err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                         ci=ci,  mew=0, alpha=.5,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color=colors[i] if subject_id in subject_ids else 'black',ax=ax[i+1])
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+2, marker='o', err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                        #  ci=0,  mew=0, alpha=1, #edit for version compatability for export
                         errorbar=None,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color='black', #colors[i] if subject_id in subject_ids else 'black',
                             ax=ax[i+1])
                ax[i+1].set_title(f'Rat {subject_id[0].upper()}',fontsize=6,y=1.01)
            if subject_id=='all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+1.5, marker='o', err_style = err_style,
#                          label=f'{subject_id}' if t==0 else None,
                         ci=ci,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=.5,# if subject_id in subject_ids else 1,
                         color='black',ax=ax[0])
#                             label='All Rats' if t==0 else None)
                ax[0].set_title(f'All Rats',fontsize=6,y=1.01)
    
            ## some stats to go with
            # compare stay trial to all switch trials in this xlim
            x1 = data_df_xlim[data_df_xlim[trial_subset] == 0][dv]
            x2 = data_df_xlim[data_df_xlim[trial_subset] != 0][dv]
            
            print(f'\nSTARTING dv: {dv} rat: {subject_id}:\n')
            result = rank_compare_2indep(x1, x2, use_t=True)
            print(f'0 vs non0\n')
            print('STATSMODELS', subject_id, trial_subset, result)
            stat, p_value = mannwhitneyu(x1, x2, alternative='two-sided')
            print(f"Wilcoxon Rank-Sum Test Statistic (SciPy): {stat}")
            print(f"P-value (SciPy): {p_value}\n")
            
#             # compare stay trial to -20 switch
#             if trial_subset == 'trials_from_next_switch':
#                 print(f'0 vs -20\n')
#                 x2 = data_df_xlim[data_df_xlim[trial_subset] == -20][dv]
#                 result = rank_compare_2indep(x1, x2, use_t=True)
#                 print('STATSMODELS', subject_id, trial_subset, result)
#                 stat, p_value = mannwhitneyu(x1, x2, alternative='two-sided')
#                 print(f"Wilcoxon Rank-Sum Test Statistic (SciPy): {stat}")
#                 print(f"P-value (SciPy): {p_value}\n")
            
#             # compare stay trial to +20 switch
#             else:
#                 print(f'0 vs +20\n')
#                 x2 = data_df_xlim[data_df_xlim[trial_subset] == 20][dv]
#                 result = rank_compare_2indep(x1, x2, use_t=True)
#                 print('STATSMODELS', subject_id, trial_subset, result)
#                 stat, p_value = mannwhitneyu(x1, x2, alternative='two-sided')
#                 print(f"Wilcoxon Rank-Sum Test Statistic (SciPy): {stat}")
#                 print(f"P-value (SciPy): {p_value}\n")
                
#             if (dv[0:6] == 'delta') & (trial_subset == 'trials_from_prior_switch'):
#                 x3 = data_df_xlim[data_df_xlim[trial_subset] == 1][dv]
#                 x4 = data_df_xlim[data_df_xlim[trial_subset] != 1][dv]
#                 print(f'1 vs non1\n')
#                 result = rank_compare_2indep(x3, x4, use_t=True)
#                 print('STATSMODELS', subject_id, trial_subset, result)
#                 stat, p_value = mannwhitneyu(x3, x4, alternative='two-sided')
#                 print(f"Wilcoxon Rank-Sum Test Statistic (SciPy): {stat}")
#                 print(f"P-value (SciPy): {p_value}\n")
            
#                 # compare stay trial to -20 switch
#             if trial_subset == 'trials_from_next_switch':
#                     x2 = data_df_xlim[data_df_xlim[trial_subset] == -20][dv]
#                     result = rank_compare_2indep(x1, x2, use_t=True)
#                     print('STATSMODELS', subject_id, trial_subset, result)
#                     stat, p_value = mannwhitneyu(x1, x2, alternative='two-sided')
#                     print(f"Wilcoxon Rank-Sum Test Statistic (SciPy): {stat}")
#                     print(f"P-value (SciPy): {p_value}")

# #                 compare stay trial to +20 switch
#             else:
#                 print(f'1 vs 20\n')
#                 x2 = data_df_xlim[data_df_xlim[trial_subset] == 20][dv]
#                 result = rank_compare_2indep(x3, x4, use_t=True)
#                 print('STATSMODELS', subject_id, trial_subset, result)
#                 stat, p_value = mannwhitneyu(x3, x4, alternative='two-sided')
#                 print(f"Wilcoxon Rank-Sum Test Statistic (SciPy): {stat}")
#                 print(f"P-value (SciPy): {p_value}\n")
                
    
    
    
    for a in range(0,6):   
        ax[a].set_xlim(trials_pre_switch_min-1, trials_post_switch_max+1)
        ax[a].spines['bottom'].set_bounds(trials_pre_switch_min,trials_post_switch_max)
#         if withlegend==True:
#             ax[a].legend(bbox_to_anchor=(1.05,.9), loc='upper left',frameon=False)
#         else:
#             pass #ax[a].get_legend().set_visible(False)
        ax[a].axvline(0, linestyle='--', color='black', zorder=0, linewidth=1)      
        if a != 0:
            ax[0].set_ylabel(f'')
    if dv == 'x':#delta_Qglobal_t_abs':
        ax[0].set_ylim(0.3,0.55)
        ax[0].set_yticks([.1,.15,.2,.25])
    elif dv== 'chosen_leaf_var_preoutcome_sess':
        ax[0].set_ylim(.02,.10)
        ax[0].set_yticks([.02,.04,.06,.08,.1])
    ax[0].set_ylabel(f'{dv}')
    ax[0].set_xlabel('Trials from Switch') 
    ax[0].set_ylabel(f'{dv}')
    plt.suptitle(f'{dv_descriptions}', y=1.1)
    sns.despine(offset=5)
    if save_fig:
        fig_name = f'addstats_betabernoulli_rat{subject_id}_only_stable_periswitch_betaDV_{dv}_exclSwitch{exclude_switches}_trialsPreSw{trials_pre_switch_min}_trialsPostSw{trials_post_switch_max}_marker{markersize}'
        plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5, dpi=300)  
    plt.show()

#### F5 B left, SF5 F

In [ ]:
# comes from sf5 ipynb, stats are lower in that nb and in f1efg nb as well

dvs = ['chosen_leaf_var_preoutcome_sess',]
dv_descriptions = {'chosen_leaf_var_preoutcome_sess': '*****var of leaf im going to pick on this trial before its outcome',}   
   
                   
assert len(dvs) == len(dv_descriptions)
# dvs = ['delta_Qleaf_t_abs_1_ago','delta_Qpatch_t_abs_1_ago','delta_Qoutofpatch_t_abs_1_ago','delta_Qglobal_t_abs_1_ago'] # add 'delta_Qleaf_t_abs' 1 AGO!


trials_post_switch_max = 20 # positive int, inclusive
trials_pre_switch_min = -20 #n egative int, inclusive
exclude_switches = False

trial_subsets = ['trials_from_next_switch','trials_from_prior_switch']

markersize=3
err_style = 'bars'
ci=95

# withlegend = False

for dv in dvs:
    fig,ax = plt.subplots(ncols=6,nrows=1,figsize=(10*TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4), sharex=True, sharey=True)
    colors = list(cm.tab20b([0,.8, .85, .1, .05]))
#     colors += 'black'
    for i,subject_id in enumerate(beta_dfs_plus_plus.keys()): #enumerate(subject_ids):
#         fig,ax = plt.subplots(ncols=1,nrows=1,figsize=(TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4))
#         print(subject_id, i)
#         print(i, subject_id)
        data_df = beta_dfs_plus_plus[subject_id].copy()

        if np.logical_and(trials_post_switch_max is not None, trials_pre_switch_min is not None):
            data_df = data_df[np.logical_or(data_df['trials_from_prior_switch']<= trials_post_switch_max,
                                             data_df['trials_from_next_switch']>= trials_pre_switch_min)]
        elif trials_post_switch_max is not None:
            data_df = data_df[data_df['trials_from_prior_switch']<= trials_post_switch_max]
        elif trials_pre_switch_min is not None:
            data_df = data_df[data_df['trials_from_next_switch']>= trials_pre_switch_min]

        if exclude_switches:
            data_df = data_df[data_df['stem_switch']==False]

        for t,trial_subset in enumerate(trial_subsets):
            # important that the "and" step happens only on one trial subset at a time
            data_df_xlim = data_df[np.logical_and(data_df[trial_subset]>=trials_pre_switch_min,
                                                 data_df[trial_subset]<=trials_post_switch_max)]
            if subject_id != 'all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize, marker=None, err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                         ci=ci,  mew=0, alpha=.5,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color=colors[i] if subject_id in subject_ids else 'black',ax=ax[i+1])
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+2, marker='o', err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                        #  ci=0,  mew=0, alpha=1,
                        errorbar=None,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color='black', #colors[i] if subject_id in subject_ids else 'black',
                             ax=ax[i+1])
                ax[i+1].set_title(f'Rat {subject_id[0].upper()}',fontsize=6,y=1.01)
            if subject_id=='all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+1.5, marker='o', err_style = err_style,
#                          label=f'{subject_id}' if t==0 else None,
                         ci=ci,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=.5,# if subject_id in subject_ids else 1,
                         color='black',ax=ax[0])
#                             label='All Rats' if t==0 else None)
                ax[0].set_title(f'All Rats',fontsize=6,y=1.01)
    for a in range(0,6):   
        ax[a].set_xlim(trials_pre_switch_min-1, trials_post_switch_max+1)
        ax[a].spines['bottom'].set_bounds(trials_pre_switch_min,trials_post_switch_max)
#         if withlegend==True:
#             ax[a].legend(bbox_to_anchor=(1.05,.9), loc='upper left',frameon=False)
#         else:
#             pass #ax[a].get_legend().set_visible(False)
        ax[a].axvline(0, linestyle='--', color='black', zorder=0, linewidth=1)      
        if a != 0:
            ax[0].set_ylabel(f'')
    if dv == 'x':#delta_Qglobal_t_abs':
        ax[0].set_ylim(0.3,0.55)
        ax[0].set_yticks([.1,.15,.2,.25])
    elif dv== 'chosen_leaf_var_preoutcome_sess':
        ax[0].set_ylim(.02,.10)
        ax[0].set_yticks([.02,.04,.06,.08,.1])
    ax[0].set_ylabel(f'{dv}')
    ax[0].set_xlabel('Trials from Switch') 
    ax[0].set_ylabel(f'{dv}')
    fig.suptitle(f'{dv_descriptions[dv]}', y=1.1)
    sns.despine(offset=5)
    if save_fig:
        fig_name = f'new_betabernoulli_rat{subject_id}_only_stable_periswitch_betaDV_{dv}_exclSwitch{exclude_switches}_trialsPreSw{trials_pre_switch_min}_trialsPostSw{trials_post_switch_max}_marker{markersize}'
        plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5, dpi=300)  
    plt.show()

In [ ]:
# comes from sf5 ipynb, stats are lower in that nb and in f1efg nb as well

dvs = ['chosen_leaf_var_preoutcome_sess',]
dv_descriptions = {'chosen_leaf_var_preoutcome_sess': '*****var of leaf im going to pick on this trial before its outcome',}   
   
                   
assert len(dvs) == len(dv_descriptions)
# dvs = ['delta_Qleaf_t_abs_1_ago','delta_Qpatch_t_abs_1_ago','delta_Qoutofpatch_t_abs_1_ago','delta_Qglobal_t_abs_1_ago'] # add 'delta_Qleaf_t_abs' 1 AGO!


trials_post_switch_max = 20 # positive int, inclusive
trials_pre_switch_min = -20 #n egative int, inclusive
exclude_switches = False

trial_subsets = ['trials_from_next_switch','trials_from_prior_switch']

markersize=3
err_style = 'bars'
ci=95

# withlegend = False

for dv in dvs:
    fig,ax = plt.subplots(ncols=6,nrows=1,figsize=(10*TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4), sharex=True, sharey=True)
    colors = list(cm.tab20b([0,.8, .85, .1, .05]))
#     colors += 'black'
    for i,subject_id in enumerate(beta_dfs_plus_plus.keys()): #enumerate(subject_ids):
#         fig,ax = plt.subplots(ncols=1,nrows=1,figsize=(TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4))
#         print(subject_id, i)
#         print(i, subject_id)
        data_df = beta_dfs_plus_plus[subject_id].copy()

        if np.logical_and(trials_post_switch_max is not None, trials_pre_switch_min is not None):
            data_df = data_df[np.logical_or(data_df['trials_from_prior_switch']<= trials_post_switch_max,
                                             data_df['trials_from_next_switch']>= trials_pre_switch_min)]
        elif trials_post_switch_max is not None:
            data_df = data_df[data_df['trials_from_prior_switch']<= trials_post_switch_max]
        elif trials_pre_switch_min is not None:
            data_df = data_df[data_df['trials_from_next_switch']>= trials_pre_switch_min]

        if exclude_switches:
            data_df = data_df[data_df['stem_switch']==False]

        for t,trial_subset in enumerate(trial_subsets):
            # important that the "and" step happens only on one trial subset at a time
            data_df_xlim = data_df[np.logical_and(data_df[trial_subset]>=trials_pre_switch_min,
                                                 data_df[trial_subset]<=trials_post_switch_max)]
            if subject_id != 'all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize, marker=None, err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                         ci=ci,  mew=0, alpha=.5,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color=colors[i] if subject_id in subject_ids else 'black',ax=ax[i+1])
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+2, marker='o', err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                        #  ci=0,  mew=0, alpha=1,
                         errorbar=None,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color='black', #colors[i] if subject_id in subject_ids else 'black',
                             ax=ax[i+1])
                ax[i+1].set_title(f'Rat {subject_id[0].upper()}',fontsize=6,y=1.01)
            if subject_id=='all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+1.5, marker='o', err_style = err_style,
#                          label=f'{subject_id}' if t==0 else None,
                         ci=ci,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=.5,# if subject_id in subject_ids else 1,
                         color='black',ax=ax[0])
#                             label='All Rats' if t==0 else None)
                ax[0].set_title(f'All Rats',fontsize=6,y=1.01)
    for a in range(0,6):   
        ax[a].set_xlim(trials_pre_switch_min-1, trials_post_switch_max+1)
        ax[a].spines['bottom'].set_bounds(trials_pre_switch_min,trials_post_switch_max)
#         if withlegend==True:
#             ax[a].legend(bbox_to_anchor=(1.05,.9), loc='upper left',frameon=False)
#         else:
#             pass #ax[a].get_legend().set_visible(False)
        ax[a].axvline(0, linestyle='--', color='black', zorder=0, linewidth=1)      
        if a != 0:
            ax[0].set_ylabel(f'')
    if dv == 'x':#delta_Qglobal_t_abs':
        ax[0].set_ylim(0.3,0.55)
        ax[0].set_yticks([.1,.15,.2,.25])
    elif dv== 'chosen_leaf_var_preoutcome_sess':
        ax[0].set_ylim(.025,.085)
        ax[0].set_yticks([.03, .04,.05,.06,.07,.08])
    ax[0].set_ylabel(f'{dv}')
    ax[0].set_xlabel('Trials from Switch') 
    ax[0].set_ylabel(f'{dv}')
    fig.suptitle(f'{dv_descriptions[dv]}', y=1.1)
    sns.despine(offset=5)
    if save_fig:
        fig_name = f'new_betabernoulli_rat{subject_id}_only_stable_periswitch_betaDV_{dv}_exclSwitch{exclude_switches}_trialsPreSw{trials_pre_switch_min}_trialsPostSw{trials_post_switch_max}_marker{markersize}'
        plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5, dpi=300)  
    plt.show()

#### F5 B right, SF5 G

In [ ]:
beta_dfs_plus5 = {}
for subject_id in subject_ids:
    beta_dfs_plus5[subject_id] = add_priorpatch_mu_updates(beta_dfs_plus4[subject_id])
    
beta_dfs_plus5['all_rats'] = pd.concat([beta_dfs_plus5[subject_id] for subject_id in subject_ids], ignore_index=True)

subject_ids = ['j16', 'chimi', 'senor', 'wilbur', 'peanut', 'all_rats']

In [ ]:
# comes from sf5 ipynb, stats are lower in that nb and in f1efg nb as well

dvs = ['global_mu_update_abs_postoutcome_sess',]
    #       'chosen_patch_mu_update_abs_postoutcome_sess',
    #       'unchosen_patches_mu_update_abs_postoutcome_sess',
    # "priorpatch_mu_update_abs_postoutcome_sess",
    # "other_unchosen_patch_mu_update_abs_postoutcome_sess"
    #      ]

dv_descriptions = {'global_mu_update_abs_postoutcome_sess': '*****summed abs global mu update as a result of t outcome',}
    #       'chosen_patch_mu_update_abs_postoutcome_sess': 'summed abs within patch mu update from t outcome',
    #       'unchosen_patches_mu_update_abs_postoutcome_sess': 'summed abs alternative patch mu update from t outcome',
    #                "priorpatch_mu_update_abs_postoutcome_sess": 'summed abs prior patch mu update from t outcome',
    # "other_unchosen_patch_mu_update_abs_postoutcome_sess":'summed abs other unchosen patch mu update from t outcome'
        #   }
        
assert len(dvs) == len(dv_descriptions)
# dvs = ['delta_Qleaf_t_abs_1_ago','delta_Qpatch_t_abs_1_ago','delta_Qoutofpatch_t_abs_1_ago','delta_Qglobal_t_abs_1_ago'] # add 'delta_Qleaf_t_abs' 1 AGO!

trials_post_switch_max = 20 # positive int, inclusive
trials_pre_switch_min = -20 #n egative int, inclusive
exclude_switches = False

trial_subsets = ['trials_from_next_switch','trials_from_prior_switch']

markersize=3
err_style = 'bars'
ci=95

# withlegend = False

for dv in dvs:
    fig,ax = plt.subplots(ncols=6,nrows=1,figsize=(10*TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4), sharex=True, sharey=True)
    colors = list(cm.tab20b([0,.8, .85, .1, .05]))
#     colors += 'black'
    for i,subject_id in enumerate(subject_ids):
#         fig,ax = plt.subplots(ncols=1,nrows=1,figsize=(TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4))
#         print(subject_id, i)
#         print(i, subject_id)
        data_df = beta_dfs_plus5[subject_id].copy()

        if np.logical_and(trials_post_switch_max is not None, trials_pre_switch_min is not None):
            data_df = data_df[np.logical_or(data_df['trials_from_prior_switch']<= trials_post_switch_max,
                                             data_df['trials_from_next_switch']>= trials_pre_switch_min)]
        elif trials_post_switch_max is not None:
            data_df = data_df[data_df['trials_from_prior_switch']<= trials_post_switch_max]
        elif trials_pre_switch_min is not None:
            data_df = data_df[data_df['trials_from_next_switch']>= trials_pre_switch_min]

        if exclude_switches:
            data_df = data_df[data_df['stem_switch']==False]

        for t,trial_subset in enumerate(trial_subsets):
            # important that the "and" step happens only on one trial subset at a time
            data_df_xlim = data_df[np.logical_and(data_df[trial_subset]>=trials_pre_switch_min,
                                                 data_df[trial_subset]<=trials_post_switch_max)]
            if subject_id != 'all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize, marker=None, err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                         ci=ci,  mew=0, alpha=.5,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color=colors[i],ax=ax[i+1])
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+2, marker='o', err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                        #  ci=0,  mew=0, alpha=1,
                        errorbar=None,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color='black', #colors[i] if subject_id in subject_ids else 'black',
                             ax=ax[i+1])
                ax[i+1].set_title(f'Rat {subject_id[0].upper()}',fontsize=6,y=1.01)
            if subject_id=='all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+1.5, marker='o', err_style = err_style,
#                          label=f'{subject_id}' if t==0 else None,
                         ci=ci,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=.5,# if subject_id in subject_ids else 1,
                         color='black',ax=ax[0])
#                             label='All Rats' if t==0 else None)
                ax[0].set_title(f'All Rats',fontsize=6,y=1.01)
    for a in range(0,6):   
        ax[a].set_xlim(trials_pre_switch_min-1, trials_post_switch_max+1)
        ax[a].spines['bottom'].set_bounds(trials_pre_switch_min,trials_post_switch_max)
#         if withlegend==True:
#             ax[a].legend(bbox_to_anchor=(1.05,.9), loc='upper left',frameon=False)
#         else:
#             pass #ax[a].get_legend().set_visible(False)
        ax[a].axvline(0, linestyle='--', color='black', zorder=0, linewidth=1)      
        if a != 0:
            ax[0].set_ylabel(f'')
    if dv == 'x':#delta_Qglobal_t_abs':
        ax[0].set_ylim(0.3,0.55)
        ax[0].set_yticks([.1,.15,.2,.25])
    elif dv== 'qGminusS_next_max':
        ax[0].set_ylim(-.35,.35)
        ax[0].set_yticks([-.3,-.2,-.1,0,.1,.2,.3])
    ax[0].set_ylabel(f'{dv}')
    ax[0].set_xlabel('Trials from Switch') 
    ax[0].set_ylabel(f'{dv}')
    fig.suptitle(f'{dv_descriptions[dv]}', y=1.1)
    sns.despine(offset=5)
    if save_fig:
        fig_name = f'betabernoulli_newvars_rat{subject_id}_only_stable_periswitch_betaDV_{dv}_exclSwitch{exclude_switches}_trialsPreSw{trials_pre_switch_min}_trialsPostSw{trials_post_switch_max}_marker{markersize}'
        plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5, dpi=300)  
    plt.show()

#### SF5 H

In [ ]:
# subset
dvs = [#'global_mu_update_abs_postoutcome_sess',
          'chosen_patch_mu_update_abs_postoutcome_sess',
          'unchosen_patches_mu_update_abs_postoutcome_sess',
#     "priorpatch_mu_update_abs_postoutcome_sess",
#     "other_unchosen_patch_mu_update_abs_postoutcome_sess"
         ]

dv_descriptions = {#'global_mu_update_abs_postoutcome_sess': '*****summed abs global mu update as a result of t outcome',
          'chosen_patch_mu_update_abs_postoutcome_sess': 'summed abs within patch mu update from t outcome',
          'unchosen_patches_mu_update_abs_postoutcome_sess': 'summed abs alternative patch mu update from t outcome',
#                    "priorpatch_mu_update_abs_postoutcome_sess": 'summed abs prior patch mu update from t outcome',
#     "other_unchosen_patch_mu_update_abs_postoutcome_sess":'summed abs other unchosen patch mu update from t outcome'
          }
assert len(dvs) == len(dv_descriptions)
# dvs = ['delta_Qleaf_t_abs_1_ago','delta_Qpatch_t_abs_1_ago','delta_Qoutofpatch_t_abs_1_ago','delta_Qglobal_t_abs_1_ago'] # add 'delta_Qleaf_t_abs' 1 AGO!

trials_post_switch_max = 20 # positive int, inclusive
trials_pre_switch_min = -20 #n egative int, inclusive
exclude_switches = False

trial_subsets = ['trials_from_next_switch','trials_from_prior_switch']

markersize=3
err_style = 'bars'
ci=95

# withlegend = False

for dv in dvs:
    fig,ax = plt.subplots(ncols=6,nrows=1,figsize=(10*TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4), sharex=True, sharey=True)
    colors = list(cm.tab20b([0,.8, .85, .1, .05]))
#     colors += 'black'
    for i,subject_id in enumerate(subject_ids):
#         fig,ax = plt.subplots(ncols=1,nrows=1,figsize=(TWO_COLUMN/3.5,(TWO_COLUMN/4)*1.4))
#         print(subject_id, i)
#         print(i, subject_id)
        data_df = beta_dfs_plus5[subject_id].copy()

        if np.logical_and(trials_post_switch_max is not None, trials_pre_switch_min is not None):
            data_df = data_df[np.logical_or(data_df['trials_from_prior_switch']<= trials_post_switch_max,
                                             data_df['trials_from_next_switch']>= trials_pre_switch_min)]
        elif trials_post_switch_max is not None:
            data_df = data_df[data_df['trials_from_prior_switch']<= trials_post_switch_max]
        elif trials_pre_switch_min is not None:
            data_df = data_df[data_df['trials_from_next_switch']>= trials_pre_switch_min]

        if exclude_switches:
            data_df = data_df[data_df['stem_switch']==False]

        for t,trial_subset in enumerate(trial_subsets):
            # important that the "and" step happens only on one trial subset at a time
            data_df_xlim = data_df[np.logical_and(data_df[trial_subset]>=trials_pre_switch_min,
                                                 data_df[trial_subset]<=trials_post_switch_max)]
            if subject_id != 'all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize, marker=None, err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                         ci=ci,  mew=0, alpha=.5,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color=colors[i],ax=ax[i+1])
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+2, marker='o', err_style = err_style,
#                          label=f'Rat {subject_id[0].upper()}' if t==0 else None,
                        #  ci=0,  mew=0, alpha=1,
                        errorbar=None,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=0.5,#if subject_id in subject_ids else 1,
                         color='black', #colors[i] if subject_id in subject_ids else 'black',
                             ax=ax[i+1])
                ax[i+1].set_title(f'Rat {subject_id[0].upper()}',fontsize=6,y=1.01)
            if subject_id=='all_rats':
                sns.lineplot(data=data_df_xlim, x=trial_subset, y=dv,
                         markersize=markersize+1.5, marker='o', err_style = err_style,
#                          label=f'{subject_id}' if t==0 else None,
                         ci=ci,  mew=0, alpha=1,
                         err_kws={'alpha':0.2, 'edgecolor':'none'} if err_style != 'bars' else None,
                         zorder=99,
                         linewidth=.5,# if subject_id in subject_ids else 1,
                         color='black',ax=ax[0])
#                             label='All Rats' if t==0 else None)
                ax[0].set_title(f'All Rats',fontsize=6,y=1.01)
    for a in range(0,6):   
        ax[a].set_xlim(trials_pre_switch_min-1, trials_post_switch_max+1)
        ax[a].spines['bottom'].set_bounds(trials_pre_switch_min,trials_post_switch_max)
#         if withlegend==True:
#             ax[a].legend(bbox_to_anchor=(1.05,.9), loc='upper left',frameon=False)
#         else:
#             pass #ax[a].get_legend().set_visible(False)
        ax[a].axvline(0, linestyle='--', color='black', zorder=0, linewidth=1)      
        if a != 0:
            ax[0].set_ylabel(f'')
    if dv == 'x':#delta_Qglobal_t_abs':
        ax[0].set_ylim(0.3,0.55)
        ax[0].set_yticks([.1,.15,.2,.25])
    elif dv== 'qGminusS_next_max':
        ax[0].set_ylim(-.35,.35)
        ax[0].set_yticks([-.3,-.2,-.1,0,.1,.2,.3])
    # shared bigger y to put on same y axis scale
    ax[0].set_ylim(0,.23)
    ax[0].set_yticks([0,.05,.1,.15,.2,])
    ax[0].set_ylabel(f'{dv}')
    ax[0].set_xlabel('Trials from Switch') 
    ax[0].set_ylabel(f'{dv}')
    fig.suptitle(f'{dv_descriptions[dv]}', y=1.1)
    sns.despine(offset=5)
    if save_fig:
        fig_name = f'betabernoulli_newvars_matchy_rat{subject_id}_only_stable_periswitch_betaDV_{dv}_exclSwitch{exclude_switches}_trialsPreSw{trials_pre_switch_min}_trialsPostSw{trials_post_switch_max}_marker{markersize}'
        plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5, dpi=300)  
    plt.show()